In [ ]:
# =====================
# TensorFlow / GPU
# =====================
import os
import tensorflow as tf

gpus = tf.config.list_physical_devices("GPU")
if gpus:
    print("GPU available:", gpus)
else:
    print("No GPU, using CPU")

if os.getenv("CUDA_VISIBLE_DEVICES") is None:
    gpu_num = 0  # 使用 CPU 可设为 ""
    os.environ["CUDA_VISIBLE_DEVICES"] = f"{gpu_num}"

os.environ["TF_CPP_MIN_LOG_LEVEL"] = "3"


# =====================
# Scientific stack
# =====================
import numpy as np
import matplotlib.pyplot as plt
from scipy.spatial.transform import Rotation as R

%matplotlib inline


# =====================
# Mitsuba / DrJit
# =====================
import mitsuba as mi
import drjit as dr


# =====================
# Sionna RT
# =====================
from sionna.rt import (
    load_scene,
    PlanarArray,
    Transmitter,
    Receiver,
    Camera,
    PathSolver,
    ITURadioMaterial,
    SceneObject,
    AntennaPattern,
    register_antenna_pattern,
)


# =====================
# Custom utils / scenes
# =====================
import sionnautils
from sionnautils.custom_scene import list_scenes, get_scene


# =====================
# Project-specific
# =====================
from Engine_V3 import Engine


# =====================
# Misc
# =====================
import json
from pathlib import Path
import yaml


In [ ]:
import yaml
import numpy as np

def normalize_config(obj):
    """Recursively normalize:
    - numeric strings (incl. scientific notation) -> float
    - 'np.pi', 'np.e', 'np.inf' -> numpy constants
    """
    if isinstance(obj, dict):
        return {k: normalize_config(v) for k, v in obj.items()}

    if isinstance(obj, list):
        return [normalize_config(v) for v in obj]

    if isinstance(obj, str):
        s = obj.strip()

        # numpy constants
        if s == "np.pi":
            return np.pi
        if s == "np.e":
            return np.e
        if s in ("np.inf", "inf"):
            return np.inf
        if s in ("-np.inf", "-inf"):
            return -np.inf

        # numeric strings (supports "15.0e9", "200.0e6", "3.5E9", etc.)
        try:
            return float(s)
        except ValueError:
            return obj  # keep as string if not numeric

    return obj  # int/float/bool/None stay as-is


with open("config.yaml", "r") as f:
    cfg = yaml.safe_load(f)

cfg = normalize_config(cfg)


In [ ]:
from sionnautils.custom_scene import list_scenes, get_scene
scenes = list_scenes()
print(scenes)

scene_path, map_data = get_scene('nyu_tandon')
for k, v in map_data.items():
    print(f'{k}: {v}')

scene = load_scene(scene_path,merge_shapes=True)

floor = scene.get('ground')
# print(f'Floor material: {floor.radio_material.name}')
floor.radio_material = ITURadioMaterial("itu_concrete",
                                "concrete",
                                thickness=0.01,
                                color=(0.5, 0.5, 0.5))

scene.remove("itu_wet_ground")

for name, obj in scene.objects.items():
    print(f'{name:<15}{obj.radio_material.name}')
# scene.render(camera=my_cam, num_samples=512)

scene.radio_materials

In [ ]:
scene.preview()

In [ ]:
# region 1
region =np.array(cfg["motion"]["region"])
region

In [ ]:
import numpy as np
import mitsuba as mi
import drjit as dr
from scipy.spatial.transform import Rotation as R

def is_inside_building_mitsuba(scene, point, direction=np.array([0.37, 0.23, 0.90]), max_hits=50):
    """
    Ray parity test: odd #hits -> inside.
    direction uses a non-axis-aligned vector to reduce degeneracy.
    Assumes watertight meshes.
    """
    point = np.asarray(point, dtype=np.float32)
    direction = np.asarray(direction, dtype=np.float32)
    direction = direction / (np.linalg.norm(direction) + 1e-12)

    ray = mi.Ray3f(o=mi.Point3f(point), d=mi.Vector3f(direction))
    scene_mi = scene._scene

    count = 0
    for _ in range(max_hits):
        si = scene_mi.ray_intersect(ray, active=True)
        if not si.is_valid():
            break
        count += 1
        ray.o = si.p + 1e-4 * ray.d  # avoid self-hit

    return (count % 2 == 1)


def ue_inside_building(scene, cfg, position, rotation, check_center_first=True):
    """
    position:  (n_ue, 3) UE center positions (world)
    rotation:  (n_ue, 3) radians, order [yaw, pitch, roll] for "zyx"
    cfg["ue"]["rx_loc_pos"]: (4, 3) UE-local RX offsets (same for all UEs)
    """
    rx_offset = np.asarray(cfg["ue"]["rx_loc_pos"], dtype=np.float32)  # (4,3)
    position = np.asarray(position, dtype=np.float32)
    rotation = np.asarray(rotation, dtype=np.float32)

    n_ue = position.shape[0]  # 用输入更稳，不依赖 cfg["ue"]["n_ue"]

    for idx_ue in range(n_ue):
        # 0) optional early reject
        if check_center_first and is_inside_building_mitsuba(scene, position[idx_ue]):
            return True

        # 1) rotate offsets (no per-RX math loop)
        r = R.from_euler("zyx", rotation[idx_ue], degrees=False)
        rx_positions = r.apply(rx_offset) + position[idx_ue]  # (4,3)

        # 2) inside test for 4 RX points (Mitsuba query is pointwise)
        #    (Still no explicit for-loop over RX in your code style)
        if any(is_inside_building_mitsuba(scene, p) for p in rx_positions):
            return True

    return False


from matplotlib.path import Path as MplPath

def is_point_in_region(region_vertices_3d, point_3d):

    point_xy = point_3d[:2]
    polygon_xy = region_vertices_3d[:, :2]
    path = MplPath(polygon_xy)
    return path.contains_point(point_xy)


In [ ]:
import gym
import numpy as np

rand_seed = cfg["motion"]["rand_seed"]

# Constants
v_intial = cfg["motion"]["v_intial"]  # initial speed

v_max = cfg["motion"]["max_speed"]
v_min = cfg["motion"]["min_speed"]
dv_min, dv_max = cfg["motion"]["dv_min"], cfg["motion"]["dv_max"]  # change in speed per seconds
dphi_min, dphi_max = np.deg2rad(cfg["motion"]["dphi_min"]), np.deg2rad(cfg["motion"]["dphi_max"])  # change in angle per step
dt = cfg["motion"]["measure_period"]
measure_time = cfg["motion"]["measure_time"]


# Action space: [Δv, Δθ]
action_space = gym.spaces.Box(low=np.array([dv_min *dt, dphi_min *dt]),
                              high=np.array([dv_max *dt, dphi_max *dt]),
                              dtype=np.float64)
action_space.seed(rand_seed)
np.random.seed(rand_seed)

In [ ]:
def wrap_pi(a):
    return (a + np.pi) % (2*np.pi) - np.pi


def initialize_ue_state(scene, cfg, region, z=1.0, max_tries=100):
    region = np.asarray(region, dtype=np.float32)
    R_speed_level = cfg["motion"]["R_speed_level"]

    for _ in range(max_tries):
        x = np.random.uniform(region[:, 0].min(), region[:, 0].max())
        y = np.random.uniform(region[:, 1].min(), region[:, 1].max())
        p0 = np.array([x, y, z], dtype=np.float32)

        if not is_point_in_region(region, p0):
            continue

        # ---- walking direction (3D, yaw only) ----
        yaw_walk0 = np.random.uniform(-np.pi, np.pi)
        walk_rot0 = np.array([yaw_walk0, 0.0, 0.0], dtype=np.float32)

        # ---- UE spin (3D) ----
        yaw_spin0 = np.random.uniform(-np.pi, np.pi)
        pitch_spin0 = np.random.uniform(-R_speed_level * np.pi,
                                         R_speed_level * np.pi)
        spin_rot0 = np.array([yaw_spin0, pitch_spin0, 0.0], dtype=np.float32)

        # ---- total UE rotation ----
        ue_rot0 = np.array([
            wrap_pi(walk_rot0[0] + spin_rot0[0]),
            spin_rot0[1],
            0.0
        ], dtype=np.float32)

        if ue_inside_building(scene, cfg, position=[p0], rotation=[ue_rot0]):
            continue

        return p0, walk_rot0, spin_rot0

    raise RuntimeError("initialize_ue_state failed")


In [ ]:

def generate_route(scene, cfg, region):

    route_list = []
    rotation_list = []   # UE total rotation [yaw, pitch, roll]
    walk_rot_list = []   # walking rotation [yaw,0,0]
    spin_rot_list = []   # UE spin rotation [yaw,pitch,0]

    R_speed_level = cfg["motion"]["R_speed_level"]

    for idx_ue in range(cfg["ue"]["n_ue"]):

        # print(f"Generating route for UE {idx_ue}...")
        route = []
        ue_rot_seq = []
        walk_rot_seq = []
        spin_rot_seq = []

        p0, walk_rot, spin_rot = initialize_ue_state(scene, cfg, region)
        x, y, z = map(float, p0)

        v = float(v_intial)
        n_steps = int(measure_time / dt)
        step = 0

        ue_rot = np.array([
            wrap_pi(walk_rot[0] + spin_rot[0]),
            spin_rot[1],
            0.0
        ], dtype=np.float32)

        # initial record
        route.append(np.array([x, y, z], dtype=np.float32))
        ue_rot_seq.append(ue_rot.copy())
        walk_rot_seq.append(walk_rot.copy())
        spin_rot_seq.append(spin_rot.copy())

        while step < n_steps-1:
            # ---- 1) walking action: update yaw only ----
            dv, dphi = action_space.sample()
            v = float(np.clip(v + dv, v_min, v_max))
            walk_rot[0] = wrap_pi(walk_rot[0] + dphi)

            # ---- 2) UE spin: yaw + pitch ----
            step_yaw = np.random.uniform(
                -R_speed_level * np.pi * dt * 2,
                R_speed_level * np.pi * dt * 2
            )
            step_pitch = np.random.uniform(
                -R_speed_level * np.pi * dt,
                R_speed_level * np.pi * dt
            )
            spin_rot[0] = wrap_pi(spin_rot[0] + step_yaw)
            spin_rot[1] = wrap_pi(spin_rot[1] + step_pitch)

            # ---- 3) compose UE rotation ----
            ue_rot = np.array([
                wrap_pi(walk_rot[0] + spin_rot[0]),
                spin_rot[1],
                0.0
            ], dtype=np.float32)

            # ---- 4) move using walking yaw only ----
            temp_x = x + v * np.cos(walk_rot[0]) * dt
            temp_y = y + v * np.sin(walk_rot[0]) * dt
            test_point = np.array([temp_x, temp_y, z], dtype=np.float32)

            if ue_inside_building(scene, cfg, position=[test_point], rotation=[ue_rot]) \
            or (not is_point_in_region(region, test_point)):
                # print(f"Step {step} rejected")
                walk_rot[0] = wrap_pi(walk_rot[0] + np.random.choice([-1, 1]) * np.pi / 6)
                continue

            # accept
            x, y = temp_x, temp_y
            route.append([x, y, z])
            ue_rot_seq.append(ue_rot.tolist())
            walk_rot_seq.append(walk_rot.tolist())
            spin_rot_seq.append(spin_rot.tolist())
            step += 1

        route_list.append(route)
        rotation_list.append(ue_rot_seq)
        walk_rot_list.append(walk_rot_seq)
        spin_rot_list.append(spin_rot_seq)

    return route_list, rotation_list, walk_rot_list, spin_rot_list


# route_list, rotation_list, walk_rot_list, spin_rot_list = generate_route(scene, cfg, region)

In [ ]:
# scene.tx_array = PlanarArray(num_rows=1,
#                              num_cols=1,
#                              vertical_spacing=0.5,
#                              horizontal_spacing=0.5,
#                              pattern="tr38901",
#                              polarization="V")

# for idx_step in range(len(route_list[0])):
#     tx = Transmitter(name=f"tx-{idx_step}",
#                     position=np.array(route_list[1][idx_step]),
#                     display_radius=5)
#     scene.remove(f"tx-{idx_step}")
#     scene.add(tx)

In [ ]:
# scene.rx_array = PlanarArray(num_rows=1,
#                               num_cols=1,
#                               vertical_spacing=0.5,
#                               horizontal_spacing=0.5,
#                               pattern="iso",
#                               polarization="V")


# for idx_step in range(len(route_list[0])):
#     # print(idx_step)
#     rx = Receiver(name=f"rx-{idx_step}",
#               position=np.array(route_list[0][idx_step]),
#               display_radius=5)
#     scene.remove(f"rx-{idx_step}")
#     scene.add(rx)

# scene.preview()

In [ ]:
import json
from pathlib import Path
from tqdm import tqdm

def to_serializable(obj):
    """Convert numpy arrays to lists recursively"""
    if isinstance(obj, dict):
        return {k: to_serializable(v) for k, v in obj.items()}
    elif isinstance(obj, list):
        return [to_serializable(item) for item in obj]
    elif hasattr(obj, 'tolist'):
        return obj.tolist()
    else:
        return obj

# Setup
output_dir = Path(cfg["route"]["folder"])
output_dir.mkdir(parents=True, exist_ok=True)

# Generate 1000 routes and save immediately
route_count = 0

for i in tqdm(range(10), desc="Generating routes"):
    r_list, rot_list, walk_list, spin_list = generate_route(scene, cfg, region)
    
    for route, rotation, walk_rot, spin_rot in zip(r_list, rot_list, walk_list, spin_list):
        route_data = {
            "route_index": route_count,
            "positions": to_serializable(route),
            "rotations": to_serializable(rotation),
            "walk_rotations": to_serializable(walk_rot),
            "spin_rotations": to_serializable(spin_rot),
            "num_steps": len(route)
        }
        
        file_path = output_dir / f"route_{route_count:04d}.json"
        with open(file_path, 'w') as f:
            json.dump(route_data, f, indent=2)
        
        route_count += 1

print(f"✅ Saved {route_count} routes to {output_dir}")


In [ ]:
# Load a single route
route_idx = 0  # change this to load different routes
route_file = output_dir / f"route_{route_idx:04d}.json"

with open(route_file, 'r') as f:
    loaded_route = json.load(f)

print(f"Route {route_idx}:")
print(f"  Steps: {loaded_route['num_steps']}")
print(f"  First position: {loaded_route['positions'][0]}")
print(f"  Last position: {loaded_route['positions'][-1]}")
print(f"  Rotations shape: {len(loaded_route['rotations'])} steps")


In [ ]:
loaded_route['rotations']